In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
path_to_class_folders="/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat"

In [3]:
import os
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

from torch.cuda.amp import autocast, GradScaler

In [4]:
paths = [
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Thalassiosira sp",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Diatom 4 (c. concavicornus)",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Diatom 3 (ditylum sp.)",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Diatom 2",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Diatom 1 (c. debilis)",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Copepod Nauplii",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Copepod",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Ciliate",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Ceratium muelleri (singular)",
    "/kaggle/input/datasets/shriharia/mp-36-underwater-microplankton-dataset/Triples_36_Classes/Triples_Raw-Subtracted-Splat/Ceratium furca (singular)"
]

SUBFOLDER = "In_focus"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 128          # 🔥 reduced
BATCH_SIZE = 8          # 🔥 reduced
EPOCHS = 20

In [5]:
class DWConv(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.conv = nn.Conv2d(ch, ch, 3, padding=1, groups=ch)
        self.bn = nn.BatchNorm2d(ch)

    def forward(self, x):
        return self.bn(self.conv(x))

In [6]:
class PlanktonDataset(Dataset):
    def __init__(self, class_paths, subfolder, transform=None):
        self.samples = []
        self.transform = transform

        self.class_names = [os.path.basename(p) for p in class_paths]
        self.class_to_idx = {cls: i for i, cls in enumerate(self.class_names)}

        for class_path in class_paths:
            label = self.class_to_idx[os.path.basename(class_path)]
            sub_path = os.path.join(class_path, subfolder)

            for img in os.listdir(sub_path):
                if img.lower().endswith(('.tif', '.tiff')):
                    self.samples.append((os.path.join(sub_path, img), label))

        print(f"Loaded {len(self.samples)} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        img = Image.open(path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label

In [7]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])

In [8]:
import torch

@torch.jit.script
def channel_shuffle(x: torch.Tensor, groups: int) -> torch.Tensor:
    b, c, h, w = x.size()
    channels_per_group = c // groups

    x = x.view(b, groups, channels_per_group, h, w)
    x = x.transpose(1, 2).contiguous()
    x = x.view(b, c, h, w)

    return x
    

In [9]:
class ShuffleBlock(nn.Module):
    def __init__(self, in_ch, out_ch, groups=4):
        super().__init__()

        mid = out_ch // 2

        self.branch1 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1),
            nn.BatchNorm2d(mid),
            nn.ReLU(),

            DWConv(mid),
            nn.ReLU()
        )

        self.branch2 = nn.Sequential(
            nn.Conv2d(in_ch, mid, 1),
            nn.BatchNorm2d(mid),
            nn.ReLU(),

            nn.Conv2d(mid, mid, 3, padding=1, groups=groups),
            nn.BatchNorm2d(mid),
            nn.ReLU(),

            nn.Conv2d(mid, mid, 1),
            nn.BatchNorm2d(mid)
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)

        out = torch.cat([b1, b2], dim=1)
        out = channel_shuffle(out, 4)

        if out.shape == x.shape:
            out = out + x

        return self.relu(out)

In [10]:
class FullModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # 🔥 Downsampling early (CRITICAL)
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        # Stage 1
        self.stage1 = nn.Sequential(*[ShuffleBlock(32, 32) for _ in range(5)])

        # Stage 2
        self.stage2 = nn.Sequential(
            ShuffleBlock(32, 64),
            nn.MaxPool2d(2),   # 🔥 reduces size
            *[ShuffleBlock(64, 64) for _ in range(4)]
        )

        # Stage 3
        self.stage3 = nn.Sequential(
            nn.MaxPool2d(2),   # 🔥 reduces size
            ShuffleBlock(64, 128)
        )

        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)

        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)

        return self.fc(x)

In [11]:
def train_model(model, train_loader, val_loader):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    scaler = GradScaler()

    for epoch in range(EPOCHS):
        model.train()
        correct, total = 0, 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            optimizer.zero_grad()

            with autocast():
                out = model(x)
                loss = criterion(out, y)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            pred = out.argmax(1)
            correct += (pred == y).sum().item()
            total += y.size(0)

        train_acc = correct / total

        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                out = model(x)
                pred = out.argmax(1)
                correct += (pred == y).sum().item()
                total += y.size(0)

        val_acc = correct / total

        print(f"Epoch {epoch+1}: Train={train_acc:.4f}, Val={val_acc:.4f}")

    return model

In [12]:
dataset = PlanktonDataset(paths, SUBFOLDER, transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

model = FullModel(num_classes=len(paths)).to(DEVICE)

model = train_model(model, train_loader, val_loader)

Loaded 7000 images


/tmp/ipykernel_57/4265429148.py:4: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_57/4265429148.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1: Train=0.2970, Val=0.4264
Epoch 2: Train=0.4263, Val=0.4129
Epoch 3: Train=0.4804, Val=0.5657
Epoch 4: Train=0.5336, Val=0.5457
Epoch 5: Train=0.5709, Val=0.6229
Epoch 6: Train=0.6207, Val=0.6371
Epoch 7: Train=0.6564, Val=0.6793
Epoch 8: Train=0.6866, Val=0.7436
Epoch 9: Train=0.7239, Val=0.6986
Epoch 10: Train=0.7366, Val=0.7150
Epoch 11: Train=0.7600, Val=0.7821
Epoch 12: Train=0.7638, Val=0.7914
Epoch 13: Train=0.7930, Val=0.8029
Epoch 14: Train=0.8127, Val=0.7643
Epoch 15: Train=0.8291, Val=0.7714
Epoch 16: Train=0.8318, Val=0.8364
Epoch 17: Train=0.8382, Val=0.8264
Epoch 18: Train=0.8523, Val=0.8121
Epoch 19: Train=0.8573, Val=0.8636
Epoch 20: Train=0.8630, Val=0.8407


In [67]:
model

FullModel(
  (stem): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
  )
  (stage1): Sequential(
    (0): ShuffleBlock(
      (branch1): Sequential(
        (0): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1))
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): DWConv(
          (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16)
          (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        )
        (4): ReLU()
      )
      (branch2): Sequential(
        (0): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1))
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU()
        (3): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)